# NSL-KDD tiny MLP for tinyinfer

In [12]:
import os, glob, json, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, confusion_matrix

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

HIDDEN       = (32, 16)
EPOCHS       = 25
BATCH        = 512
LR           = 3e-3
WD           = 1e-4
DROPOUT      = 0.1
INPUT_NOISE  = 0.05
LABEL_SMOOTH = 0.05
CLIP_STD     = 10.0      # standardized inputs are clipped to +-CLIP_STD (train and test)
N_SPLITS     = 5
N_SEEDS      = 3
THRESH_MODE  = "fpr"     # "fpr": fixed false-alarm budget | "fbeta": maximize F-beta on OOF
TARGET_FPR   = 0.05
BETA         = 1.0

SEEDS = list(range(SEED, SEED + N_SEEDS))

In [13]:
COLUMNS_ALL = [
    "duration","protocol_type","service","flag","src_bytes","dst_bytes","land",
    "wrong_fragment","urgent","hot","num_failed_logins","logged_in","num_compromised",
    "root_shell","su_attempted","num_root","num_file_creations","num_shells",
    "num_access_files","num_outbound_cmds","is_host_login","is_guest_login",
    "count","srv_count","serror_rate","srv_serror_rate","rerror_rate","srv_rerror_rate",
    "same_srv_rate","diff_srv_rate","srv_diff_host_rate","dst_host_count",
    "dst_host_srv_count","dst_host_same_srv_rate","dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate","dst_host_srv_diff_host_rate","dst_host_serror_rate",
    "dst_host_srv_serror_rate","dst_host_rerror_rate","dst_host_srv_rerror_rate",
    "label","difficulty",
]

ATTACK_CATEGORY = {
    **dict.fromkeys(["neptune","back","land","pod","smurf","teardrop","mailbomb",
                     "apache2","processtable","udpstorm","worm"], "DoS"),
    **dict.fromkeys(["ipsweep","nmap","portsweep","satan","mscan","saint"], "Probe"),
    **dict.fromkeys(["ftp_write","guess_passwd","imap","multihop","phf","spy",
                     "warezclient","warezmaster","sendmail","named","snmpgetattack",
                     "snmpguess","xlock","xsnoop","httptunnel"], "R2L"),
    **dict.fromkeys(["buffer_overflow","loadmodule","perl","rootkit","ps",
                     "sqlattack","xterm"], "U2R"),
}

def find_dataset(name):
    for root in ["/kaggle/input", ".", "/media/hdd/cybersec_ai/datasets"]:
        hits = glob.glob(os.path.join(root, "**", name), recursive=True)
        if hits:
            return sorted(hits, key=len)[0]
    raise FileNotFoundError(name)

train = pd.read_csv(find_dataset("KDDTrain+.txt"), names=COLUMNS_ALL).drop(columns=["difficulty"])
test  = pd.read_csv(find_dataset("KDDTest+.txt"),  names=COLUMNS_ALL).drop(columns=["difficulty"])

unknown = (set(train.label) | set(test.label)) - set(ATTACK_CATEGORY) - {"normal"}
print("Labels missing from category map:", unknown or "none")

y_train = (train.label != "normal").astype(np.int64).values
y_test  = (test.label  != "normal").astype(np.int64).values
test_cat = test.label.map(lambda l: "normal" if l == "normal" else ATTACK_CATEGORY.get(l, "other")).values
print("Train attack rate: %.3f | Test attack rate: %.3f" % (y_train.mean(), y_test.mean()))
print("Test-only attack types:", sorted(set(test.label) - set(train.label)))

Labels missing from category map: none
Train attack rate: 0.465 | Test attack rate: 0.569
Test-only attack types: ['apache2', 'httptunnel', 'mailbomb', 'mscan', 'named', 'processtable', 'ps', 'saint', 'sendmail', 'snmpgetattack', 'snmpguess', 'sqlattack', 'udpstorm', 'worm', 'xlock', 'xsnoop', 'xterm']


## Features

In [14]:
CAT_DROP = ["protocol_type", "service", "flag", "label"]
NUM_COLS = [c for c in COLUMNS_ALL if c not in CAT_DROP + ["difficulty"]]
NUM_COLS = [c for c in NUM_COLS if train[c].std() > 0]
LOG_COLS = [c for c in ["duration","src_bytes","dst_bytes","hot","num_compromised",
                        "num_root","num_file_creations","count","srv_count",
                        "dst_host_count","dst_host_srv_count"] if c in NUM_COLS]
PROTOS = ["tcp", "udp", "icmp"]

def build(df):
    X = df[NUM_COLS].astype(np.float64).copy()
    X[LOG_COLS] = np.log1p(X[LOG_COLS].clip(lower=0))
    for p in PROTOS:
        X["proto_" + p] = (df.protocol_type == p).astype(np.float64)
    return X

def scale(X, sc):
    return np.clip(sc.transform(X), -CLIP_STD, CLIP_STD).astype(np.float32)

Xtr_df, Xte_df = build(train), build(test)
FEATURES = list(Xtr_df.columns)
Xtr_raw = Xtr_df.values.astype(np.float32)
Xte_raw = Xte_df.values.astype(np.float32)
D = Xtr_raw.shape[1]
print("Input dim:", D)

Input dim: 40


## Grouped CV

In [15]:
rng = np.random.RandomState(SEED)
groups = np.where(y_train == 0,
                  np.char.add("normal_", rng.randint(0, 20, len(train)).astype(str)),
                  train.label.values)
folds = list(GroupKFold(n_splits=N_SPLITS).split(Xtr_raw, y_train, groups))

for k, (tr, va) in enumerate(folds):
    lab = pd.Series(train.label.values[va])
    top = lab[lab != "normal"].value_counts().head(3).to_dict()
    print("fold %d: n=%d attack_rate=%.3f top_attacks=%s" % (k, len(va), y_train[va].mean(), top))

def sigmoid(s):
    return 1.0 / (1.0 + np.exp(-s))

def metrics(y, s, thr):
    pred = s > thr
    tp = int((pred & (y == 1)).sum()); fp = int((pred & (y == 0)).sum())
    fn = int((~pred & (y == 1)).sum()); tn = int((~pred & (y == 0)).sum())
    prec = tp / max(tp + fp, 1); rec = tp / max(tp + fn, 1)
    return dict(acc=(tp + tn) / len(y), prec=prec, rec=rec, fpr=fp / max(fp + tn, 1),
                f1=2 * prec * rec / max(prec + rec, 1e-9), auc=roc_auc_score(y, s))

def show(name, m):
    print("%-40s acc=%.4f prec=%.4f rec=%.4f fpr=%.4f f1=%.4f auc=%.4f" %
          (name, m["acc"], m["prec"], m["rec"], m["fpr"], m["f1"], m["auc"]))

fold 0: n=41214 attack_rate=1.000 top_attacks={'neptune': 41214}
fold 1: n=21203 attack_rate=0.367 top_attacks={'satan': 3633, 'smurf': 2646, 'nmap': 1493}
fold 2: n=21186 attack_rate=0.365 top_attacks={'ipsweep': 3599, 'portsweep': 2931, 'back': 956}
fold 3: n=21186 attack_rate=0.046 top_attacks={'teardrop': 892, 'guess_passwd': 53, 'land': 18}
fold 4: n=21184 attack_rate=0.044 top_attacks={'warezclient': 890, 'warezmaster': 20, 'rootkit': 10}


## Model

In [16]:
def make_mlp(d, hidden=HIDDEN):
    layers, p = [], d
    for h in hidden:
        layers += [nn.Linear(p, h), nn.ReLU(), nn.Dropout(DROPOUT)]
        p = h
    layers.append(nn.Linear(p, 2))
    return nn.Sequential(*layers)

def train_mlp(X, y, seed, hidden=HIDDEN, epochs=EPOCHS):
    torch.manual_seed(seed)
    Xt, yt = torch.tensor(X), torch.tensor(y)
    m = make_mlp(X.shape[1], hidden)
    opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=WD)
    steps_per_epoch = int(np.ceil(len(X) / BATCH))
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=epochs * steps_per_epoch)
    for _ in range(epochs):
        m.train()
        perm = torch.randperm(len(Xt))
        for i in range(0, len(Xt), BATCH):
            idx = perm[i:i + BATCH]
            xb = Xt[idx] + INPUT_NOISE * torch.randn(len(idx), Xt.shape[1])
            loss = F.cross_entropy(m(xb), yt[idx], label_smoothing=LABEL_SMOOTH)
            opt.zero_grad(); loss.backward(); opt.step(); sch.step()
    m.eval()
    return m

# score = logit1 - logit0. The device decision is: attack if score > threshold.
def mlp_score(m, X):
    with torch.no_grad():
        o = m(torch.tensor(X))
        return (o[:, 1] - o[:, 0]).numpy().astype(np.float64)

def mlp_fit_score(Xa, ya, Xb, seed):
    sc = StandardScaler().fit(Xa)
    m = train_mlp(scale(Xa, sc), ya, seed)
    return mlp_score(m, scale(Xb, sc))

def make_hgb(seed):
    return HistGradientBoostingClassifier(max_depth=6, learning_rate=0.1, max_iter=200,
                                          l2_regularization=1.0, random_state=seed)

def hgb_score(clf, X):
    p = np.clip(clf.predict_proba(X)[:, 1], 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

def hgb_fit_score(Xa, ya, Xb, seed):
    return hgb_score(make_hgb(seed).fit(Xa, ya), Xb)

def oof_scores(fit_score, seed):
    s = np.zeros(len(y_train))
    for tr, va in folds:
        s[va] = fit_score(Xtr_raw[tr], y_train[tr], Xtr_raw[va], seed)
    return s

def thr_fpr(y, s, fpr=TARGET_FPR):
    return float(np.quantile(s[y == 0], 1 - fpr))

def thr_fbeta(y, s, beta=BETA):
    best_t, best_f = None, -1.0
    for t in np.quantile(s, np.linspace(0.005, 0.995, 400)):
        pred = s > t
        tp = (pred & (y == 1)).sum(); fp = (pred & (y == 0)).sum(); fn = (~pred & (y == 1)).sum()
        p = tp / max(tp + fp, 1); r = tp / max(tp + fn, 1)
        f = (1 + beta**2) * p * r / max(beta**2 * p + r, 1e-9)
        if f > best_f:
            best_t, best_f = float(t), float(f)
    return best_t, best_f

def pick_thr(y, s):
    return thr_fpr(y, s) if THRESH_MODE == "fpr" else thr_fbeta(y, s)[0]

## OOF comparison

In [17]:
oof_mlp = [oof_scores(mlp_fit_score, s) for s in SEEDS]
oof_hgb = oof_scores(hgb_fit_score, SEED)

print("Grouped OOF (threshold picked on the same OOF scores, mode=%s):" % THRESH_MODE)
for s, o in zip(SEEDS, oof_mlp):
    show("TinyMLP seed %d" % s, metrics(y_train, o, pick_thr(y_train, o)))
show("HistGB reference", metrics(y_train, oof_hgb, pick_thr(y_train, oof_hgb)))

Grouped OOF (threshold picked on the same OOF scores, mode=fpr):
TinyMLP seed 42                          acc=0.9570 prec=0.9438 rec=0.9651 fpr=0.0500 f1=0.9543 auc=0.9846
TinyMLP seed 43                          acc=0.9474 prec=0.9427 rec=0.9445 fpr=0.0500 f1=0.9436 auc=0.9821
TinyMLP seed 44                          acc=0.9625 prec=0.9445 rec=0.9768 fpr=0.0500 f1=0.9603 auc=0.9827
HistGB reference                         acc=0.9405 prec=0.9418 rec=0.9297 fpr=0.0500 f1=0.9357 auc=0.9878


## Deployed model

In [18]:
scaler = StandardScaler().fit(Xtr_raw)
Xtr_s = scale(Xtr_raw, scaler)
Xte_s = scale(Xte_raw, scaler)
print("Clipped inputs: train %.4f%% | test %.4f%%" %
      (100 * (np.abs(Xtr_s) >= CLIP_STD - 1e-4).mean(), 100 * (np.abs(Xte_s) >= CLIP_STD - 1e-4).mean()))

models = [train_mlp(Xtr_s, y_train, s) for s in SEEDS]
te_scores = [mlp_score(m, Xte_s) for m in models]
thrs = [pick_thr(y_train, o) for o in oof_mlp]
res = [metrics(y_test, s, t) for s, t in zip(te_scores, thrs)]

for seed, t, r in zip(SEEDS, thrs, res):
    show("seed %d (thr %.2f)" % (seed, t), r)
print("Mean +/- std over %d seeds:" % N_SEEDS)
for k in ["acc", "prec", "rec", "fpr", "f1", "auc"]:
    v = np.array([r[k] for r in res])
    print("  %-5s %.4f +/- %.4f" % (k, v.mean(), v.std()))

m_dep, s_dep, THR = models[0], te_scores[0], thrs[0]
r_dep = res[0]

Clipped inputs: train 0.0826% | test 0.1569%
seed 42 (thr -3.22)                      acc=0.8752 prec=0.9193 rec=0.8559 fpr=0.0993 f1=0.8865 auc=0.9039
seed 43 (thr -3.21)                      acc=0.8844 prec=0.9349 rec=0.8567 fpr=0.0789 f1=0.8941 auc=0.9235
seed 44 (thr -3.21)                      acc=0.8748 prec=0.9176 rec=0.8570 fpr=0.1016 f1=0.8863 auc=0.9066
Mean +/- std over 3 seeds:
  acc   0.8782 +/- 0.0044
  prec  0.9239 +/- 0.0078
  rec   0.8565 +/- 0.0005
  fpr   0.0933 +/- 0.0102
  f1    0.8890 +/- 0.0036
  auc   0.9113 +/- 0.0087


## Report

In [19]:
pred_dep = (s_dep > THR).astype(int)
print("Deployed: seed %d | mode %s | threshold %.3f (logit diff) = %.3f (prob)" %
      (SEEDS[0], THRESH_MODE, THR, sigmoid(THR)))
show("deployed", r_dep)
cm = confusion_matrix(y_test, pred_dep)
print("TN %d | FP %d\nFN %d | TP %d" % (cm[0, 0], cm[0, 1], cm[1, 0], cm[1, 1]))

print("\nPer-category recall:")
for c in ["DoS", "Probe", "R2L", "U2R", "other"]:
    mk = test_cat == c
    if mk.any():
        print("  %-6s n=%5d recall=%.3f" % (c, mk.sum(), pred_dep[mk].mean()))
print("  normal n=%5d specificity=%.3f" % ((test_cat == "normal").sum(), 1 - pred_dep[test_cat == "normal"].mean()))

tab = (pd.DataFrame({"label": test.label.values, "hit": pred_dep})
       .query("label != 'normal'").groupby("label").agg(n_test=("hit", "size"), recall=("hit", "mean")))
tab["n_train"] = tab.index.map(train.label.value_counts()).fillna(0).astype(int)
print("\nTop test attack types (n_train = examples seen in training):")
print(tab.sort_values("n_test", ascending=False).head(15).round(3).to_string())

print("\nFalse-alarm budget -> test result (threshold from OOF):")
for b in [0.01, 0.02, 0.05, 0.10]:
    t = thr_fpr(y_train, oof_mlp[0], b)
    r = metrics(y_test, s_dep, t)
    print("  budget %.2f | thr %.2f | test fpr %.3f | recall %.3f | acc %.3f" % (b, t, r["fpr"], r["rec"], r["acc"]))

rs = np.random.RandomState(SEED)
accs, recs = [], []
for _ in range(500):
    i = rs.randint(0, len(y_test), len(y_test))
    accs.append((pred_dep[i] == y_test[i]).mean())
    recs.append(pred_dep[i][y_test[i] == 1].mean())
print("\n95%% bootstrap CI (test sampling only): acc [%.4f, %.4f] | recall [%.4f, %.4f]" %
      (*np.percentile(accs, [2.5, 97.5]), *np.percentile(recs, [2.5, 97.5])))

Deployed: seed 42 | mode fpr | threshold -3.219 (logit diff) = 0.038 (prob)
deployed                                 acc=0.8752 prec=0.9193 rec=0.8559 fpr=0.0993 f1=0.8865 auc=0.9039
TN 8747 | FP 964
FN 1849 | TP 10984

Per-category recall:
  DoS    n= 7460 recall=0.949
  Probe  n= 2421 recall=0.990
  R2L    n= 2885 recall=0.506
  U2R    n=   67 recall=0.716
  normal n= 9711 specificity=0.901

Top test attack types (n_train = examples seen in training):
               n_test  recall  n_train
label                                 
neptune          4657   1.000    41214
guess_passwd     1231   0.389       53
mscan             996   0.977        0
warezmaster       944   0.850       20
apache2           737   0.996        0
satan             735   1.000     3633
processtable      685   0.877        0
smurf             665   1.000     2646
back              359   1.000      956
snmpguess         331   0.130        0
saint             319   0.997        0
mailbomb          293   0.007      

## HistGB reference

In [20]:
hgb = make_hgb(SEED).fit(Xtr_raw, y_train)
s_hgb = hgb_score(hgb, Xte_raw)
show("HistGB @ own OOF threshold", metrics(y_test, s_hgb, pick_thr(y_train, oof_hgb)))
show("TinyMLP deployed", r_dep)

HistGB @ own OOF threshold               acc=0.8986 prec=0.9380 rec=0.8799 fpr=0.0768 f1=0.9080 auc=0.9652
TinyMLP deployed                         acc=0.8752 prec=0.9193 rec=0.8559 fpr=0.0993 f1=0.8865 auc=0.9039


## Int8 check (simulated)

In [21]:
lin = [l for l in m_dep if isinstance(l, nn.Linear)]
Ws = [l.weight.detach().numpy().astype(np.float64) for l in lin]
Bs = [l.bias.detach().numpy().astype(np.float64) for l in lin]

def fq(x, s):
    return np.clip(np.round(x / s), -128, 127) * s

def pscale(a, pct=99.9):
    return max(np.percentile(np.abs(a), pct), 1e-8) / 127.0

IN_SCALE = CLIP_STD / 127.0
W_SCALES = [float(np.abs(W).max() / 127.0) for W in Ws]

def float_forward(X):
    h = X.astype(np.float64)
    for i, (W, b) in enumerate(zip(Ws, Bs)):
        h = h @ W.T + b
        if i < len(Ws) - 1:
            h = np.maximum(h, 0)
    return h

cal = Xtr_s[np.random.RandomState(SEED).choice(len(Xtr_s), min(20000, len(Xtr_s)), replace=False)]
ACT_SCALES, h = [], cal.astype(np.float64)
for i, (W, b) in enumerate(zip(Ws, Bs)):
    h = h @ W.T + b
    if i < len(Ws) - 1:
        h = np.maximum(h, 0)
    ACT_SCALES.append(float(pscale(h)))

def quant_forward(X):
    h = fq(X.astype(np.float64), IN_SCALE)
    for i, (W, b) in enumerate(zip(Ws, Bs)):
        h = h @ fq(W, W_SCALES[i]).T + b
        if i < len(Ws) - 1:
            h = np.maximum(h, 0)
        h = fq(h, ACT_SCALES[i])
    return h

lf, lq = float_forward(Xte_s), quant_forward(Xte_s)
assert np.allclose(lf[:, 1] - lf[:, 0], s_dep, atol=1e-3), "numpy forward != torch forward"
pf = ((lf[:, 1] - lf[:, 0]) > THR).astype(int)
pq = ((lq[:, 1] - lq[:, 0]) > THR).astype(int)
print("float32 acc: %.4f | int8-sim acc: %.4f | decision agreement: %.4f" %
      ((pf == y_test).mean(), (pq == y_test).mean(), (pf == pq).mean()))

float32 acc: 0.8752 | int8-sim acc: 0.8786 | decision agreement: 0.9938


## Export

In [24]:
OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."

rr = np.random.RandomState(SEED)
ref_idx = np.concatenate([rr.choice(np.where(y_test == 0)[0], 32, replace=False),
                          rr.choice(np.where(y_test == 1)[0], 32, replace=False)])

arrays = {}
for i, (W, b) in enumerate(zip(Ws, Bs)):
    arrays["W%d" % i] = W.astype(np.float32)
    arrays["b%d" % i] = b.astype(np.float32)
    arrays["W%d_q" % i] = np.clip(np.round(W / W_SCALES[i]), -128, 127).astype(np.int8)
arrays["w_scales"]   = np.array(W_SCALES, dtype=np.float32)
arrays["act_scales"] = np.array(ACT_SCALES, dtype=np.float32)
arrays["in_scale"]   = np.array([IN_SCALE], dtype=np.float32)
arrays["ref_inputs"]      = Xte_s[ref_idx]
arrays["ref_labels"]      = y_test[ref_idx].astype(np.int32)
arrays["ref_logits"]      = lf[ref_idx].astype(np.float32)
arrays["ref_logits_int8sim"] = lq[ref_idx].astype(np.float32)
np.savez(os.path.join(OUT, "tinyinfer_weights.npz"), **arrays)

scaler_params = {
    "mean_": scaler.mean_.tolist(),
    "scale_": scaler.scale_.tolist(),
    "feature_names": FEATURES,
    "log1p_cols": LOG_COLS,
    "protocols": PROTOS,
    "clip_std": CLIP_STD,
}
to_py = lambda d: {k: float(v) for k, v in d.items()}
seed_stats = {k: dict(mean=float(np.mean([r[k] for r in res])), std=float(np.std([r[k] for r in res])))
              for k in ["acc", "prec", "rec", "fpr", "f1", "auc"]}
meta = dict(
    architecture="-".join(str(x) for x in [D, *HIDDEN, 2]),
    features=FEATURES, scaler_params=scaler_params,
    threshold_mode=THRESH_MODE, target_fpr=TARGET_FPR,
    logit_diff_threshold=float(THR), threshold_prob=float(sigmoid(THR)),
    decision="attack if (logit1 - logit0) > logit_diff_threshold",
    test_accuracy=float(r_dep["acc"]), deployed_metrics=to_py(r_dep), seed_stats=seed_stats,
    deployed_seed=SEEDS[0],
    preprocessing="x -> log1p on log1p_cols -> proto one-hot -> (x-mean)/scale -> clip to +-clip_std",
    int8=dict(scheme="symmetric per-tensor, simulated", in_scale=IN_SCALE,
              w_scales=W_SCALES, act_scales=ACT_SCALES),
)
with open(os.path.join(OUT, "tinyinfer_meta.json"), "w") as f:
    json.dump(meta, f, indent=2)

torch.save({
    "model_state_dict": m_dep.state_dict(),
    "scaler_params":    scaler_params,
    "features":         FEATURES,
    "architecture":     meta["architecture"],
    "test_accuracy":    meta["test_accuracy"],
    "threshold":        float(THR),
}, os.path.join(OUT, "tinyinfer_mlp.pth"))
print("Saved: tinyinfer_mlp.pth")

n_params = sum(W.size + b.size for W, b in zip(Ws, Bs))
print("Saved to", OUT, "| params:", n_params, "(%.1f KB f32 | %.1f KB int8)" % (n_params * 4 / 1024, n_params / 1024))

Saved: tinyinfer_mlp.pth
Saved to /kaggle/working | params: 1874 (7.3 KB f32 | 1.8 KB int8)


## Preprocessing check

In [23]:
def prep_row(row, meta):
    sp = meta["scaler_params"]
    vals = []
    for name in sp["feature_names"]:
        if name.startswith("proto_"):
            v = float(row["protocol_type"] == name[len("proto_"):])
        else:
            v = float(row[name])
            if name in sp["log1p_cols"]:
                v = np.log1p(max(v, 0.0))
        vals.append(v)
    z = (np.array(vals) - np.array(sp["mean_"])) / np.array(sp["scale_"])
    return np.clip(z, -sp["clip_std"], sp["clip_std"])

n_chk = 300
chk = np.stack([prep_row(test.iloc[i], meta) for i in range(n_chk)])
err = np.abs(chk - Xte_s[:n_chk]).max()
print("Max abs error, meta-only preprocessing vs pipeline: %.2e" % err)
assert err < 1e-3, "meta.json is not enough to reproduce preprocessing"

Max abs error, meta-only preprocessing vs pipeline: 5.42e-07
